# **Assignment 08: Stable WAN 2.1**

**Available:** Oct 16, 2025 3:00pm until Oct 25, 2025 11:59pm

**Details**
- Dataset: https://www.kaggle.com/datasets/sharjeelmazhar/human-activity-recognition-video-dataset
- Train LoRA for WAN 2.1 1.3G model (not the 14G model, too big)​
- Generate videos for human activity recognition, 10 per category.​
- Grader will judge the quality and gives grade


## **Setup**

In [ ]:
## Import Libraries

# Set CUDA_VISIBLE_DEVICES to make both GPUs visible
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

# Install all packages for from the requirements.txt
%pip install -r requirements.txt

import torch
import torch.nn as nn
import torchvision
import datasets
import cv2
import matplotlib.pyplot as plt
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch import optim
from tqdm.notebook import tqdm
from torchinfo import summary
import einops
import PIL
import numpy as np
import pandas as pd
# Use a pipeline as a high-level helper
from transformers import pipeline
import time
import psutil
import gc
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import json
from collections import defaultdict
import numpy as np

# Authorize Huggingface account
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/.env')

# Get Hugging Face token
hf_token = os.getenv('HUGGINGFACE_HUB_TOKEN') or os.getenv('HF_TOKEN')

if hf_token:
    print("Found Hugging Face token in environment variables")
    
    
    from huggingface_hub import login, whoami
    
    try:
        # Login to Hugging Face Hub
        login(token=hf_token)
        
        # Verify login by getting user info
        user_info = whoami()
        print(f"Successfully authenticated with Hugging Face!")
        print(f"Logged in as: {user_info['name']}")
        
        # Set the token as environment variable for other libraries
        os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token
        os.environ['HF_TOKEN'] = hf_token
        
    except Exception as e:
        print(f"Authentication failed: {e}")
        print("Will proceed without pre-trained models if needed")
        hf_token = None
else:
    print("No Hugging Face token found in .env file")
    print("Please add HUGGINGFACE_HUB_TOKEN=your_token_here to your .env file")
    hf_token = None


# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")


In [ ]:
# GPU Setup 
# Comprehensive GPU diagnostics
print("\n=== GPU Diagnostics ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print("\n=== All Available GPUs ===")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}:")
        print(f"  Name: {props.name}")
        print(f"  Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"  Multi-processor count: {props.multi_processor_count}")
        print(f"  Compute Capability: {props.major}.{props.minor}")
        print()

# Device selection with preference for cuda:1 (A6000) -> cuda:0 (4090) -> mps (Apple Silicon) -> cpu
if torch.cuda.is_available() and torch.cuda.device_count() > 1:
    device = torch.device('cuda:1')  # This should now be your A6000!
    print(f"Using GPU 1: {torch.cuda.get_device_name(1)}")
elif torch.cuda.is_available():
    device = torch.device('cuda:0')
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print("Using Apple Silicon MPS")
else:
    device = torch.device('cpu')
    print("Using CPU")

print(f"Selected device: {device}")

# If no logs folder exists, create one
if not os.path.exists("logs"):
    os.makedirs("logs")

# If no checkpoints folder exists, create one
if not os.path.exists("checkpoints"):
    os.makedirs("checkpoints")

# If no data folder exists, create one
if not os.path.exists("data"):
    os.makedirs("data")

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process()
    return process.memory_info().rss / 1024 / 1024

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    elif device.type == 'mps':
        # MPS doesn't have direct memory monitoring like CUDA
        # Return 0 as a placeholder
        return 0
    return 0

In [ ]:
# Function to clear memory for both CUDA and MPS
def clear_memory():
    """Clear CPU and GPU memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print(f"Cleared memory. Current CPU memory usage: {get_memory_usage():.2f} MB, GPU memory usage: {get_gpu_memory_usage():.2f} MB")

## **Import Model**

In [ ]:
## Source: https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B

import os
import shutil

# Clone the WAN 2.1 repository only if it doesn't already exist
if not os.path.exists('Wan2.1'):
    print("Cloning WAN 2.1 repository...")
    !git clone https://github.com/Wan-Video/Wan2.1.git
    
    # Deactivate the cloned repository by removing .git folder
    git_folder = os.path.join('Wan2.1', '.git')
    if os.path.exists(git_folder):
        print("Deactivating cloned repository (removing .git folder)...")
        shutil.rmtree(git_folder)
        print("Wan2.1 is no longer a git repository")
    else:
        print("No .git folder found in cloned repository")
else:
    print("WAN 2.1 repository already exists, skipping clone.")

# Change to the Wan2.1 directory
os.chdir('Wan2.1')
print(f"Changed to directory: {os.getcwd()}")

# Install requirements (ensure torch >= 2.4.0)
!pip install -r requirements.txt

# Download the model from Hugging Face only if it doesn't already exist
if not os.path.exists('./Wan2.1-T2V-1.3B'):
    print("Downloading WAN 2.1 model...")
    !huggingface-cli download Wan-AI/Wan2.1-T2V-1.3B --local-dir ./Wan2.1-T2V-1.3B
else:
    print("WAN 2.1 model already exists, skipping download.")

## **Implement LORA into WAN 2.1**

In [ ]:
# Implement LORA into WAN 2.1 model:

import sys
sys.path.append('./Wan2.1')

from wan.modules.model import WanModel
from peft import LoraConfig, get_peft_model, TaskType
import torch.nn as nn
from typing import List, Dict, Any

def create_lora_wan_model(model_config: Dict[str, Any], lora_config: Dict[str, Any] = None):
    """
    Create a WAN 2.1 model with LoRA adapters
    
    Args:
        model_config: Configuration dictionary for the WAN model
        lora_config: Configuration dictionary for LoRA adapters
    
    Returns:
        WAN model with LoRA adapters attached
    """
    
    # Default LoRA configuration if not provided
    if lora_config is None:
        lora_config = {
            "r": 16,  # rank
            "lora_alpha": 32,  # scaling factor
            "lora_dropout": 0.1,  # dropout probability
            "target_modules": [
                # Self-attention layers
                "q", "k", "v", "o",  # in WanSelfAttention
                # Cross-attention layers  
                "k_img", "v_img",   # in WanI2VCrossAttention (if applicable)
                # FFN layers
                "ffn.0", "ffn.2",   # first and third layers in FFN
                # Projection layers
                "head.head",        # output head
                "text_embedding.0", "text_embedding.2"  # text embedding layers
            ]
        }
    
    # Create the base WAN model
    model = WanModel(**model_config)
    
    # Configure LoRA
    peft_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,  # Using feature extraction as base task type
        inference_mode=False,
        r=lora_config["r"],
        lora_alpha=lora_config["lora_alpha"], 
        lora_dropout=lora_config["lora_dropout"],
        target_modules=lora_config["target_modules"],
        bias="none"  # Don't adapt bias parameters
    )
    
    # Apply LoRA to the model
    model = get_peft_model(model, peft_config)
    
    return model

def print_trainable_parameters(model):
    """
    Print the number of trainable parameters in the model
    """
    trainable_params = 0
    all_param = 0
    
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    
    print(f"Trainable params: {trainable_params:,} || "
          f"All params: {all_param:,} || "
          f"Trainable%: {100 * trainable_params / all_param:.2f}")

# Example usage - Create WAN 2.1 model with LoRA
print("Creating WAN 2.1 model with LoRA adapters...")

# Model configuration for WAN 2.1 1.3B (smaller version)
wan_config = {
    "model_type": "t2v",  # text-to-video
    "patch_size": (1, 2, 2),
    "text_len": 512,
    "in_dim": 16,
    "dim": 2048,        # Reduced from 2048 for 1.3B version
    "ffn_dim": 8192,    # Reduced accordingly  
    "freq_dim": 256,
    "text_dim": 4096,
    "out_dim": 16,
    "num_heads": 16,    # Reduced from 32
    "num_layers": 24,   # Reduced from 32 for 1.3B version
    "window_size": (-1, -1),
    "qk_norm": True,
    "cross_attn_norm": True,
    "eps": 1e-6
}

# LoRA configuration - you can adjust these hyperparameters
lora_config = {
    "r": 16,              # Rank - higher = more parameters but better adaptation
    "lora_alpha": 32,     # Scaling factor - typically 2x the rank
    "lora_dropout": 0.1,  # Dropout for regularization
    "target_modules": [
        # Self-attention components in each block
        "blocks.*.self_attn.q",
        "blocks.*.self_attn.k", 
        "blocks.*.self_attn.v",
        "blocks.*.self_attn.o",
        # Cross-attention components
        "blocks.*.cross_attn.q",
        "blocks.*.cross_attn.k",
        "blocks.*.cross_attn.v", 
        "blocks.*.cross_attn.o",
        # FFN components
        "blocks.*.ffn.0",  # First linear layer in FFN
        "blocks.*.ffn.2",  # Second linear layer in FFN
        # Head projection
        "head.head",
        # Text embedding layers
        "text_embedding.0",
        "text_embedding.2"
    ]
}

try:
    # Create the model with LoRA
    wan_lora_model = create_lora_wan_model(wan_config, lora_config)
    
    # Move to appropriate device
    wan_lora_model = wan_lora_model.to(device)
    
    print(f"Successfully created WAN 2.1 model with LoRA adapters on {device}")
    
    # Print parameter information
    print("\n" + "="*50)
    print("MODEL PARAMETER SUMMARY")
    print("="*50)
    print_trainable_parameters(wan_lora_model)
    
    # Print model structure (first few layers)
    print(f"\nModel structure preview:")
    print(f"Model type: {wan_lora_model.config.model_type}")
    print(f"Number of layers: {wan_lora_model.config.num_layers}")
    print(f"Hidden dimension: {wan_lora_model.config.dim}")
    print(f"Number of attention heads: {wan_lora_model.config.num_heads}")
    
    # Show some LoRA adapter info
    print(f"\nLoRA Configuration:")
    print(f"Rank (r): {lora_config['r']}")
    print(f"Alpha: {lora_config['lora_alpha']}")
    print(f"Dropout: {lora_config['lora_dropout']}")
    
    print(f"\nWAN 2.1 with LoRA is ready for training!")
    
except Exception as e:
    print(f" Error creating WAN model with LoRA: {str(e)}")
    print("This might be due to model architecture changes or missing dependencies.")
    import traceback
    traceback.print_exc()

# Clear some memory
clear_memory()

## **Dataset Preparation**

In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import json
from pathlib import Path
import random
from tqdm import tqdm
import matplotlib.pyplot as plt

class HARVideoDataset(Dataset):
    """
    Human Action Recognition Video Dataset for WAN 2.1 training
    Processes videos into frames suitable for text-to-video generation training
    """
    
    def __init__(self, data_dir, split='train', max_frames=16, frame_size=(224, 224), 
                 train_ratio=0.8, random_seed=42):
        """
        Initialize the HAR Video Dataset
        
        Args:
            data_dir: Path to HAR_Video_Dataset directory
            split: 'train', 'val', or 'test'
            max_frames: Maximum number of frames to extract per video
            frame_size: Target frame size (height, width)
            train_ratio: Ratio of data to use for training
            random_seed: Random seed for reproducibility
        """
        self.data_dir = Path(data_dir)
        self.split = split
        self.max_frames = max_frames
        self.frame_size = frame_size
        self.random_seed = random_seed
        
        # Define action classes
        self.classes = [
            "Clapping",
            "Meet and Split", 
            "Sitting",
            "Standing Still",
            "Walking",
            "Walking While Reading Book",
            "Walking While Using Phone"
        ]
        
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        # Load and split dataset
        self.video_paths, self.labels, self.prompts = self._load_dataset()
        
        print(f"Loaded {len(self.video_paths)} videos for {split} split")
        print(f"Classes: {self.classes}")
        print(f"Class distribution: {dict(zip(self.classes, [self.labels.count(i) for i in range(len(self.classes))]))}")
    
    def _load_dataset(self):
        """Load video paths and create train/val splits"""
        all_videos = []
        all_labels = []
        all_prompts = []
        
        # Collect all videos
        for class_name in self.classes:
            class_dir = self.data_dir / class_name
            if not class_dir.exists():
                print(f"Warning: Class directory {class_dir} not found")
                continue
                
            video_files = list(class_dir.glob("*.mp4"))
            class_label = self.class_to_idx[class_name]
            
            for video_file in video_files:
                all_videos.append(str(video_file))
                all_labels.append(class_label)
                # Create descriptive prompts for each action
                prompt = self._create_prompt(class_name)
                all_prompts.append(prompt)
        
        # Split dataset
        if len(all_videos) == 0:
            raise ValueError("No videos found in dataset")
        
        # Create train/val/test splits
        train_videos, temp_videos, train_labels, temp_labels, train_prompts, temp_prompts = train_test_split(
            all_videos, all_labels, all_prompts, 
            train_size=0.7, random_state=self.random_seed, stratify=all_labels
        )
        
        val_videos, test_videos, val_labels, test_labels, val_prompts, test_prompts = train_test_split(
            temp_videos, temp_labels, temp_prompts,
            train_size=0.5, random_state=self.random_seed, stratify=temp_labels
        )
        
        # Return appropriate split
        if self.split == 'train':
            return train_videos, train_labels, train_prompts
        elif self.split == 'val':
            return val_videos, val_labels, val_prompts
        else:  # test
            return test_videos, test_labels, test_prompts
    
    def _create_prompt(self, class_name):
        """Create descriptive text prompts for each action class"""
        prompt_templates = {
            "Clapping": [
                "A person clapping their hands together",
                "Person applauding by clapping hands",
                "Human clapping with both hands",
                "Someone clapping their hands rhythmically"
            ],
            "Meet and Split": [
                "People meeting and then separating", 
                "Group of people gathering then splitting apart",
                "Individuals coming together and then going separate ways",
                "People meeting briefly then walking away"
            ],
            "Sitting": [
                "A person sitting down",
                "Person sitting in a seated position",
                "Human sitting calmly",
                "Someone sitting still"
            ],
            "Standing Still": [
                "A person standing motionless",
                "Person standing still without moving",
                "Human standing in place",
                "Someone standing upright and stationary"
            ],
            "Walking": [
                "A person walking forward",
                "Person walking at normal pace",
                "Human walking naturally",
                "Someone taking steps while walking"
            ],
            "Walking While Reading Book": [
                "A person walking while reading a book",
                "Person reading and walking simultaneously",
                "Human walking and holding a book to read",
                "Someone walking while focused on reading"
            ],
            "Walking While Using Phone": [
                "A person walking while using their phone",
                "Person walking and looking at mobile device",
                "Human walking while texting on phone", 
                "Someone walking while using smartphone"
            ]
        }
        
        return random.choice(prompt_templates[class_name])
    
    def _extract_frames(self, video_path):
        """Extract frames from video file"""
        cap = cv2.VideoCapture(video_path)
        frames = []
        
        # Get video properties
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        
        if total_frames == 0:
            cap.release()
            return None
        
        # Calculate frame indices to extract
        if total_frames <= self.max_frames:
            # Use all frames if video is short
            frame_indices = list(range(total_frames))
        else:
            # Sample frames uniformly across the video
            frame_indices = np.linspace(0, total_frames-1, self.max_frames, dtype=int)
        
        # Extract frames
        for frame_idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            
            if ret:
                # Convert BGR to RGB
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                # Resize frame
                frame = cv2.resize(frame, self.frame_size)
                # Normalize to [0, 1]
                frame = frame.astype(np.float32) / 255.0
                frames.append(frame)
        
        cap.release()
        
        if len(frames) == 0:
            return None
        
        # Pad or truncate to max_frames
        while len(frames) < self.max_frames:
            frames.append(frames[-1])  # Repeat last frame
        frames = frames[:self.max_frames]
        
        # Convert to tensor (T, H, W, C) -> (C, T, H, W)
        frames_tensor = torch.tensor(np.array(frames))
        frames_tensor = frames_tensor.permute(3, 0, 1, 2)  # (C, T, H, W)
        
        return frames_tensor
    
    def __len__(self):
        return len(self.video_paths)
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        prompt = self.prompts[idx]
        
        # Extract frames
        frames = self._extract_frames(video_path)
        
        if frames is None:
            # Return a random valid sample if current video fails
            return self.__getitem__(random.randint(0, len(self) - 1))
        
        return {
            'frames': frames,
            'label': label,
            'prompt': prompt,
            'class_name': self.classes[label],
            'video_path': video_path
        }

def create_har_dataloaders(data_dir, batch_size=4, max_frames=16, frame_size=(224, 224), num_workers=2):
    """
    Create train, validation, and test dataloaders for HAR dataset
    
    Args:
        data_dir: Path to HAR_Video_Dataset directory
        batch_size: Batch size for dataloaders
        max_frames: Maximum frames per video
        frame_size: Target frame size (H, W)
        num_workers: Number of worker processes
    
    Returns:
        train_loader, val_loader, test_loader, class_names
    """
    
    # Create datasets
    train_dataset = HARVideoDataset(
        data_dir=data_dir,
        split='train',
        max_frames=max_frames,
        frame_size=frame_size
    )
    
    val_dataset = HARVideoDataset(
        data_dir=data_dir,
        split='val', 
        max_frames=max_frames,
        frame_size=frame_size
    )
    
    test_dataset = HARVideoDataset(
        data_dir=data_dir,
        split='test',
        max_frames=max_frames,
        frame_size=frame_size
    )
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader, test_loader, train_dataset.classes

# Initialize the dataset
print(" Initializing Human Action Recognition Dataset for WAN 2.1 training...")

data_dir = "/mnt/Storage02/SoftwareDev/CAP_6411_Assignments/Aug2025/01_Assignments/08_WAN_2_1/data/HAR_Video_Dataset"

# Check if dataset exists
if not os.path.exists(data_dir):
    print(f" Dataset directory not found: {data_dir}")
else:
    print(f"Found dataset directory: {data_dir}")
    
    # Create dataloaders with appropriate settings for WAN 2.1
    try:
        train_loader, val_loader, test_loader, class_names = create_har_dataloaders(
            data_dir=data_dir,
            batch_size=2,  # Small batch size for video data
            max_frames=16,  # 16 frames per video clip
            frame_size=(256, 256),  # Standard resolution for video models
            num_workers=2
        )
        
        print(f"\nDataset Statistics:")
        print(f"   Classes: {len(class_names)} ({', '.join(class_names)})")
        print(f"   Training samples: {len(train_loader.dataset)}")
        print(f"   Validation samples: {len(val_loader.dataset)}")
        print(f"   Test samples: {len(test_loader.dataset)}")
        print(f"   Batch size: {train_loader.batch_size}")
        print(f"   Max frames per video: 16")
        print(f"   Frame resolution: 256x256")
        
        # Test loading a batch
        print("\nTesting data loading...")
        sample_batch = next(iter(train_loader))
        
        print(f"   Batch shape - Frames: {sample_batch['frames'].shape}")
        print(f"   Batch labels: {sample_batch['label']}")
        print(f"   Sample prompts: {sample_batch['prompt'][:2]}")  # Show first 2 prompts
        
        print(f"\nDataset successfully prepared for WAN 2.1 training!")
        print(f"   - Video frames are normalized to [0,1] range")
        print(f"   - Frame format: (Batch, Channels, Time, Height, Width)")
        print(f"   - Text prompts generated for each action class") 
        print(f"   - Train/Val/Test splits created with stratification")
        
        # Save dataset info for reference
        dataset_info = {
            'classes': class_names,
            'num_classes': len(class_names),
            'train_size': len(train_loader.dataset),
            'val_size': len(val_loader.dataset), 
            'test_size': len(test_loader.dataset),
            'max_frames': 16,
            'frame_size': [256, 256],
            'batch_size': 2
        }

        os.chdir('..')
        filepath = 'data/dataset_info.json'

        # Create dataset_info.json file if it doesn't exist
        os.makedirs(os.path.dirname(filepath), exist_ok=True)

        with open(filepath, 'w') as f:
            json.dump(dataset_info, f, indent=2)

        print(f"\nDataset info saved to '{filepath}'")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
    except Exception as e:
        print(f" Error creating dataloaders: {str(e)}")
        import traceback
        traceback.print_exc()

# Clear memory after dataset preparation
clear_memory()

## **Setup Training**

In [ ]:
## Setup Training of LoRA-augmented WAN 2.1 Model

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
import json
from datetime import datetime

class WAN21Trainer:
    """
    Trainer class for WAN 2.1 with LoRA adapters
    """
    
    def __init__(self, model, train_loader, val_loader, config, device):
        """
        Initialize the trainer
        
        Args:
            model: WAN 2.1 model wrapper (contains LoRA-enabled WAN inside)
            train_loader: Training data loader
            val_loader: Validation data loader
            config: Training configuration dictionary
            device: Device to train on
        """
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.device = device
        
        # Setup optimizer - only optimize trainable parameters (LoRA adapters)
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        print(f"  Found {len(trainable_params)} trainable parameter groups")
        
        self.optimizer = AdamW(
            trainable_params,
            lr=config['learning_rate'],
            weight_decay=config['weight_decay'],
            betas=(0.9, 0.999)
        )
        
        # Setup learning rate scheduler
        if config['scheduler'] == 'cosine':
            self.scheduler = CosineAnnealingLR(
                self.optimizer,
                T_max=config['num_epochs'],
                eta_min=config['learning_rate'] * 0.01
            )
        elif config['scheduler'] == 'onecycle':
            self.scheduler = OneCycleLR(
                self.optimizer,
                max_lr=config['learning_rate'],
                epochs=config['num_epochs'],
                steps_per_epoch=len(train_loader),
                pct_start=0.3
            )
        else:
            self.scheduler = None
        
        # Setup loss function (MSE for video generation)
        self.criterion = nn.MSELoss()
        
        # Training history
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'learning_rates': []
        }
        
        # Best model tracking
        self.best_val_loss = float('inf')
        self.best_epoch = 0
        
        print("Trainer initialized successfully")
        print(f"  - Optimizer: AdamW (lr={config['learning_rate']}, wd={config['weight_decay']})")
        print(f"  - Scheduler: {config['scheduler']}")
        print(f"  - Loss function: MSE")
        print(f"  - Number of epochs: {config['num_epochs']}")
    
    def train_epoch(self, epoch):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        num_batches = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}/{self.config['num_epochs']} [Train]")
        
        for batch_idx, batch in enumerate(pbar):
            # Move data to device
            frames = batch['frames'].to(self.device)  # (B, C, T, H, W)
            prompts = batch['prompt']  # List of text prompts
            
            # Forward pass
            try:
                # Zero gradients
                self.optimizer.zero_grad()
                
                # Generate random timesteps for diffusion training
                timesteps = torch.randint(
                    0, 1000, (frames.size(0),), 
                    device=self.device, dtype=torch.long
                )
                
                # Add noise to frames (diffusion forward process)
                noise = torch.randn_like(frames)
                noisy_frames = frames + noise * (timesteps.float().view(-1, 1, 1, 1, 1) / 1000.0)
                
                # Get model predictions using wrapper's forward method
                # This calls: wrapper.forward(x, t, prompts) -> no PEFT interference
                outputs = self.model(noisy_frames, timesteps, prompts)
                
                # Calculate loss (predicting the noise)
                loss = self.criterion(outputs, noise)
                
                # Backward pass
                loss.backward()
                
                # Gradient clipping
                if self.config.get('grad_clip', 0) > 0:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), 
                        self.config['grad_clip']
                    )
                
                # Optimizer step
                self.optimizer.step()
                
                # Update learning rate (if using OneCycle)
                if self.config['scheduler'] == 'onecycle':
                    self.scheduler.step()
                
                # Update metrics
                total_loss += loss.item()
                num_batches += 1
                
                # Update progress bar
                pbar.set_postfix({
                    'loss': f"{loss.item():.4f}",
                    'avg_loss': f"{total_loss/num_batches:.4f}",
                    'lr': f"{self.optimizer.param_groups[0]['lr']:.6f}"
                })
                
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"\n WARNING: OOM at batch {batch_idx}. Skipping batch and clearing cache...")
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                else:
                    raise e
        
        avg_loss = total_loss / num_batches if num_batches > 0 else 0
        return avg_loss
    
    def validate(self, epoch):
        """Validate the model"""
        self.model.eval()
        total_loss = 0
        num_batches = 0
        
        pbar = tqdm(self.val_loader, desc=f"Epoch {epoch}/{self.config['num_epochs']} [Val]")
        
        with torch.no_grad():
            for batch in pbar:
                # Move data to device
                frames = batch['frames'].to(self.device)
                prompts = batch['prompt']
                
                try:
                    # Generate random timesteps for diffusion training
                    timesteps = torch.randint(
                        0, 1000, (frames.size(0),), 
                        device=self.device, dtype=torch.long
                    )
                    
                    # Add noise to frames (diffusion forward process)
                    noise = torch.randn_like(frames)
                    noisy_frames = frames + noise * (timesteps.float().view(-1, 1, 1, 1, 1) / 1000.0)
                    
                    # Forward pass using wrapper
                    outputs = self.model(noisy_frames, timesteps, prompts)
                    
                    # Calculate loss (predicting the noise)
                    loss = self.criterion(outputs, noise)
                    
                    # Update metrics
                    total_loss += loss.item()
                    num_batches += 1
                    
                    # Update progress bar
                    pbar.set_postfix({
                        'loss': f"{loss.item():.4f}",
                        'avg_loss': f"{total_loss/num_batches:.4f}"
                    })
                    
                except RuntimeError as e:
                    if "out of memory" in str(e):
                        print(f"\n WARNING: OOM in validation. Skipping batch...")
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        continue
                    else:
                        raise e
        
        avg_loss = total_loss / num_batches if num_batches > 0 else 0
        return avg_loss
    
    def save_checkpoint(self, epoch, filename):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict() if self.scheduler else None,
            'history': self.history,
            'config': self.config,
            'best_val_loss': self.best_val_loss,
            'best_epoch': self.best_epoch
        }
        
        torch.save(checkpoint, filename)
        print(f"Checkpoint saved: {filename}")
    
    def load_checkpoint(self, filename):
        """Load model checkpoint"""
        checkpoint = torch.load(filename, map_location=self.device)
        
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        if self.scheduler and checkpoint['scheduler_state_dict']:
            self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        self.history = checkpoint['history']
        self.best_val_loss = checkpoint['best_val_loss']
        self.best_epoch = checkpoint['best_epoch']
        
        print(f"Checkpoint loaded: {filename}")
        return checkpoint['epoch']
    
    def train(self):
        """Main training loop"""
        print("\n" + "="*60)
        print("STARTING TRAINING")
        print("="*60)
        
        start_time = datetime.now()
        
        for epoch in range(1, self.config['num_epochs'] + 1):
            # Train
            train_loss = self.train_epoch(epoch)
            self.history['train_loss'].append(train_loss)
            
            # Validate
            val_loss = self.validate(epoch)
            self.history['val_loss'].append(val_loss)
            
            # Update learning rate (if using Cosine)
            if self.config['scheduler'] == 'cosine':
                self.scheduler.step()
            
            # Record learning rate
            current_lr = self.optimizer.param_groups[0]['lr']
            self.history['learning_rates'].append(current_lr)
            
            # Print epoch summary
            print(f"\n{'='*60}")
            print(f"Epoch {epoch}/{self.config['num_epochs']} Summary:")
            print(f"  Train Loss: {train_loss:.4f}")
            print(f"  Val Loss:   {val_loss:.4f}")
            print(f"  LR:         {current_lr:.6f}")
            
            # Save best model
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.best_epoch = epoch
                best_model_path = Path(self.config['checkpoint_dir']) / 'best_model.pth'
                self.save_checkpoint(epoch, best_model_path)
                print(f"  🌟 New best model! (Val Loss: {val_loss:.4f})")
            
            # Save regular checkpoint
            if epoch % self.config['save_every'] == 0:
                checkpoint_path = Path(self.config['checkpoint_dir']) / f'checkpoint_epoch_{epoch}.pth'
                self.save_checkpoint(epoch, checkpoint_path)
            
            # Clear memory
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            print(f"{'='*60}\n")
        
        end_time = datetime.now()
        training_time = end_time - start_time
        
        print("\n" + "="*60)
        print("TRAINING COMPLETE")
        print("="*60)
        print(f"Total training time: {training_time}")
        print(f"Best epoch: {self.best_epoch}")
        print(f"Best validation loss: {self.best_val_loss:.4f}")
        
        # Save final training history
        history_path = Path(self.config['checkpoint_dir']) / 'training_history.json'
        with open(history_path, 'w') as f:
            json.dump(self.history, f, indent=2)
        print(f"Training history saved to: {history_path}")
        
        return self.history
    
    def plot_training_history(self):
        """Plot training history"""
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        # Plot losses
        axes[0].plot(self.history['train_loss'], label='Train Loss', marker='o')
        axes[0].plot(self.history['val_loss'], label='Val Loss', marker='s')
        axes[0].axvline(x=self.best_epoch-1, color='r', linestyle='--', 
                       label=f'Best Epoch ({self.best_epoch})')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training and Validation Loss')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Plot learning rate
        axes[1].plot(self.history['learning_rates'], marker='o', color='green')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Learning Rate')
        axes[1].set_title('Learning Rate Schedule')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save plot
        plot_path = Path(self.config['checkpoint_dir']) / 'training_history.png'
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        print(f"Training history plot saved to: {plot_path}")
        
        plt.show()

# Training configuration
training_config = {
    'num_epochs': 50,
    'learning_rate': 1e-4,
    'weight_decay': 0.01,
    'scheduler': 'cosine',  # 'cosine' or 'onecycle' or None
    'grad_clip': 1.0,
    'save_every': 5,  # Save checkpoint every N epochs
    'checkpoint_dir': './checkpoints'
}

# Create checkpoint directory
os.makedirs(training_config['checkpoint_dir'], exist_ok=True)

print("Initializing WAN 2.1 LoRA Trainer...")
print(f"\n Training Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

# Initialize trainer with the FIXED model
try:
    trainer = WAN21Trainer(
        model=wan_lora_model,  # This is now the wrapper with LoRA inside
        train_loader=train_loader,
        val_loader=val_loader,
        config=training_config,
        device=device
    )
    
    print("\nTrainer ready!")
    print("   Model has custom forward signature (no PEFT interference)")
    print("   LoRA adapters will be trained")
    print("   Text encoder is frozen")
    print("\nTo start training, call: trainer.train()")
    print("After training, visualize results with: trainer.plot_training_history()")
    
except Exception as e:
    print(f" Error initializing trainer: {str(e)}")
    import traceback
    traceback.print_exc()


## **CRITICAL FIX: WAN Model Wrapper**

The issue is that WAN model expects a specific input format that's incompatible with PEFT's expectations. We need to create a wrapper.

In [ ]:
## Create WAN Model Wrapper for Training - FIXED VERSION

# The WAN model expects specific input format:
# - x: List of video tensors (not batched)
# - t: timesteps
# - context: List of TEXT EMBEDDINGS (not raw strings)
# - seq_len: maximum sequence length

# CRITICAL FIX: Don't apply PEFT to the wrapper - apply it directly to the WAN model
# Then wrap the PEFT model in our custom wrapper

from transformers import T5EncoderModel, T5Tokenizer
import torch.nn as nn

print("🔧 Creating WAN model wrapper for proper training...")

# Load T5 for text encoding (WAN uses T5-XXL)
print("\nLoading T5 text encoder...")
try:
    text_encoder = T5EncoderModel.from_pretrained("google/t5-v1_1-xxl", torch_dtype=torch.float16)
    text_tokenizer = T5Tokenizer.from_pretrained("google/t5-v1_1-xxl")
    text_encoder = text_encoder.to(device)
    text_encoder.eval()  # Keep frozen
    print("   T5-XXL loaded successfully")
except Exception as e:
    print(f"    Could not load T5-XXL, trying T5-XL: {e}")
    try:
        text_encoder = T5EncoderModel.from_pretrained("google/t5-v1_1-xl", torch_dtype=torch.float16)
        text_tokenizer = T5Tokenizer.from_pretrained("google/t5-v1_1-xl")
        text_encoder = text_encoder.to(device)
        text_encoder.eval()
        print("   T5-XL loaded as fallback")
    except:
        print("    Could not load T5, using random embeddings for testing")
        text_encoder = None
        text_tokenizer = None

class WanModelWrapper(nn.Module):
    """
    Wrapper around WAN model to handle batch processing and text encoding
    This wrapper should NOT be wrapped by PEFT - apply PEFT to inner wan_model first
    """
    
    def __init__(self, wan_model, text_encoder, text_tokenizer, max_seq_len=512):
        super().__init__()
        self.wan_model = wan_model  # This can be a PEFT-wrapped WAN model
        self.text_encoder = text_encoder
        self.text_tokenizer = text_tokenizer
        self.max_seq_len = max_seq_len
        
        # Store config for compatibility (unwrap if PEFT model)
        if hasattr(wan_model, 'config'):
            self.config = wan_model.config
        elif hasattr(wan_model, 'base_model') and hasattr(wan_model.base_model, 'config'):
            self.config = wan_model.base_model.config
        else:
            self.config = None
        
    def encode_text(self, prompts):
        """Encode text prompts to embeddings"""
        if self.text_encoder is None:
            # Return dummy embeddings if no encoder
            batch_size = len(prompts)
            device = next(self.wan_model.parameters()).device
            return [torch.randn(512, 4096, device=device) 
                    for _ in range(batch_size)]
        
        # Tokenize
        with torch.no_grad():
            tokens = self.text_tokenizer(
                prompts,
                padding='max_length',
                max_length=512,
                truncation=True,
                return_tensors='pt'
            ).to(self.text_encoder.device)
            
            # Encode
            embeddings = self.text_encoder(
                input_ids=tokens['input_ids'],
                attention_mask=tokens['attention_mask']
            ).last_hidden_state
            
            # Convert to list of tensors
            return [embeddings[i] for i in range(embeddings.size(0))]
    
    def forward(self, x, t, prompts):
        """
        Forward pass that converts from batch format to WAN's expected format
        
        Args:
            x: Batched video tensor [B, C, T, H, W]
            t: Timesteps [B]
            prompts: List of text strings
            
        Returns:
            Batched output tensor [B, C, T, H, W]
        """
        batch_size = x.size(0)
        
        # Convert batched tensor to list of tensors
        x_list = [x[i] for i in range(batch_size)]
        
        # Encode text prompts
        context_list = self.encode_text(prompts)
        
        # Calculate max sequence length needed
        seq_len = self.max_seq_len
        
        # Call WAN model (which may have LoRA adapters inside)
        output_list = self.wan_model(
            x=x_list,
            t=t,
            context=context_list,
            seq_len=seq_len
        )
        
        # Convert list of tensors back to batch
        output = torch.stack(output_list, dim=0)
        
        return output

print("\nApplying LoRA to base WAN model FIRST...")

# Get the base WAN model (before any wrapping)
from wan.modules.model import WanModel

# Recreate base WAN model
wan_config = {
    "model_type": "t2v",
    "patch_size": (1, 2, 2),
    "text_len": 512,
    "in_dim": 16,
    "dim": 2048,
    "ffn_dim": 8192,
    "freq_dim": 256,
    "text_dim": 4096,
    "out_dim": 16,
    "num_heads": 16,
    "num_layers": 24,
    "window_size": (-1, -1),
    "qk_norm": True,
    "cross_attn_norm": True,
    "eps": 1e-6
}

base_wan = WanModel(**wan_config)

# Apply LoRA configuration to identify target modules
from peft import LoraConfig, get_peft_model, TaskType

# Build LoRA target modules dynamically
lora_target_modules = []
num_blocks = 24

for i in range(num_blocks):
    # Target the linear layers in each block
    lora_target_modules.extend([
        f"blocks.{i}.self_attn.q",
        f"blocks.{i}.self_attn.k",
        f"blocks.{i}.self_attn.v",
        f"blocks.{i}.self_attn.o",
        f"blocks.{i}.ffn.0",
        f"blocks.{i}.ffn.2",
    ])

print(f"   Targeting {len(lora_target_modules)} modules with LoRA")

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=lora_target_modules,
    bias="none"
)

try:
    # Apply LoRA to base WAN model
    print("   🔧 Applying LoRA adapters to WAN model...")
    wan_with_lora = get_peft_model(base_wan, peft_config)
    wan_with_lora = wan_with_lora.to(device)
    
    print("   LoRA applied successfully to WAN model!")
    print_trainable_parameters(wan_with_lora)
    
    # NOW wrap the LoRA-enabled WAN model
    print("\n Wrapping LoRA-enabled WAN model...")
    wan_lora_wrapped = WanModelWrapper(
        wan_model=wan_with_lora,
        text_encoder=text_encoder,
        text_tokenizer=text_tokenizer,
        max_seq_len=4096
    )
    
    print("   Wrapper created around LoRA model")
    
    # Update global model variable
    wan_lora_model = wan_lora_wrapped
    
    print("\nWAN model ready for training!")
    print("   - LoRA adapters: Applied to inner WAN model")
    print("   - Wrapper: Handles batch conversion and text encoding")
    print("   - Forward signature: (x, t, prompts) ← Custom, no PEFT interference")
    
except Exception as e:
    print(f"    Could not apply LoRA: {e}")
    print(" Showing detailed error:")
    import traceback
    traceback.print_exc()
    
    print("\n     Using wrapper without LoRA...")
    wan_lora_wrapped = WanModelWrapper(
        wan_model=base_wan.to(device),
        text_encoder=text_encoder,
        text_tokenizer=text_tokenizer,
        max_seq_len=4096
    )
    wan_lora_model = wan_lora_wrapped

# Clear memory
clear_memory()


In [ ]:
## Start Training

# Start the training process
print(" Starting training loop...")
print("This may take several hours depending on your GPU and dataset size.")
print("\nTraining will:")
print("  Save best model based on validation loss")
print("  Save checkpoints every 5 epochs")
print("  Track training history")
print("  Handle OOM errors gracefully")
print("\n" + "="*60 + "\n")

try:
    # Run training
    history = trainer.train()
    
    # Plot training history
    print("\nGenerating training plots...")
    trainer.plot_training_history()
    
    print("\nTraining completed successfully!")
    print(f"   Best model saved at: {training_config['checkpoint_dir']}/best_model.pth")
    
except KeyboardInterrupt:
    print("\n Training interrupted by user")
    print(f"   Last checkpoint saved at epoch {len(trainer.history['train_loss'])}")
    
    # Save interrupted training state
    interrupted_path = Path(training_config['checkpoint_dir']) / 'interrupted_checkpoint.pth'
    trainer.save_checkpoint(len(trainer.history['train_loss']), interrupted_path)
    
except Exception as e:
    print(f"\n Training failed with error: {str(e)}")
    import traceback
    traceback.print_exc()
    
    # Try to save emergency checkpoint
    try:
        emergency_path = Path(training_config['checkpoint_dir']) / 'emergency_checkpoint.pth'
        trainer.save_checkpoint(len(trainer.history['train_loss']), emergency_path)
        print(f"Emergency checkpoint saved to: {emergency_path}")
    except:
        print("Could not save emergency checkpoint")

finally:
    # Clean up memory
    clear_memory()
    print("\n Memory cleared")

In [ ]:
## Training Utilities - Resume Training & Monitor Progress

def resume_training(checkpoint_path, trainer):
    """
    Resume training from a checkpoint
    
    Args:
        checkpoint_path: Path to checkpoint file
        trainer: WAN21Trainer instance
    
    Returns:
        Starting epoch
    """
    print(f" Loading checkpoint from: {checkpoint_path}")
    start_epoch = trainer.load_checkpoint(checkpoint_path)
    print(f"Resuming from epoch {start_epoch}")
    return start_epoch

def monitor_training_progress(checkpoint_dir='./checkpoints'):
    """
    Monitor current training progress
    
    Args:
        checkpoint_dir: Directory containing checkpoints
    """
    checkpoint_dir = Path(checkpoint_dir)
    
    # Check for training history
    history_file = checkpoint_dir / 'training_history.json'
    if history_file.exists():
        with open(history_file, 'r') as f:
            history = json.load(f)
        
        print("Training Progress Summary:")
        print(f"  Completed epochs: {len(history['train_loss'])}")
        print(f"  Latest train loss: {history['train_loss'][-1]:.4f}")
        print(f"  Latest val loss: {history['val_loss'][-1]:.4f}")
        print(f"  Current learning rate: {history['learning_rates'][-1]:.6f}")
        
        # Find best epoch
        best_val_loss = min(history['val_loss'])
        best_epoch = history['val_loss'].index(best_val_loss) + 1
        print(f"\n  Best validation loss: {best_val_loss:.4f} (Epoch {best_epoch})")
        
        # Plot current progress
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
        axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
        axes[0].axvline(x=best_epoch-1, color='r', linestyle='--', 
                       label=f'Best Epoch ({best_epoch})')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training Progress')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        axes[1].plot(history['learning_rates'], marker='o', color='green')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Learning Rate')
        axes[1].set_title('Learning Rate Schedule')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
    else:
        print(" No training history found. Training hasn't started yet.")
    
    # List available checkpoints
    checkpoints = sorted(checkpoint_dir.glob('*.pth'))
    if checkpoints:
        print(f"\nAvailable checkpoints:")
        for ckpt in checkpoints:
            size_mb = ckpt.stat().st_size / (1024 * 1024)
            print(f"  - {ckpt.name} ({size_mb:.2f} MB)")
    else:
        print("\n No checkpoints found yet.")

def calculate_training_eta(history, total_epochs, checkpoint_dir='./checkpoints'):
    """
    Calculate estimated time to completion
    
    Args:
        history: Training history dictionary
        total_epochs: Total number of epochs to train
        checkpoint_dir: Directory containing checkpoints
    """
    if len(history['train_loss']) < 2:
        print(" Not enough training data to estimate ETA")
        return
    
    completed_epochs = len(history['train_loss'])
    remaining_epochs = total_epochs - completed_epochs
    
    # Try to estimate time per epoch from checkpoint timestamps
    checkpoint_dir = Path(checkpoint_dir)
    checkpoints = sorted(checkpoint_dir.glob('checkpoint_epoch_*.pth'))
    
    if len(checkpoints) >= 2:
        # Calculate average time between checkpoints
        times = [ckpt.stat().st_mtime for ckpt in checkpoints[-5:]]  # Last 5 checkpoints
        if len(times) >= 2:
            avg_time_per_checkpoint = (times[-1] - times[0]) / (len(times) - 1)
            epochs_per_checkpoint = 5  # Based on save_every=5
            avg_time_per_epoch = avg_time_per_checkpoint / epochs_per_checkpoint
            
            eta_seconds = avg_time_per_epoch * remaining_epochs
            eta_hours = eta_seconds / 3600
            
            print(f" Training ETA:")
            print(f"  Completed: {completed_epochs}/{total_epochs} epochs")
            print(f"  Remaining: {remaining_epochs} epochs")
            print(f"  Average time per epoch: {avg_time_per_epoch/60:.2f} minutes")
            print(f"  Estimated time to completion: {eta_hours:.2f} hours")
    else:
        print(f" Completed {completed_epochs}/{total_epochs} epochs")
        print(" Not enough checkpoints to estimate ETA accurately")

# Example usage:
print(" Training Utilities Loaded")
print("\nAvailable functions:")
print("  1. monitor_training_progress() - View current training status")
print("  2. resume_training(checkpoint_path, trainer) - Resume from checkpoint")
print("  3. calculate_training_eta(history, total_epochs) - Estimate completion time")
print("\nExample: monitor_training_progress('./checkpoints')")